# 01 — Synthetic NQ Futures Data

Generates a synthetic multi-year daily price series for NQ (Nasdaq-100 E-mini) futures with three distinct market regimes — **bull**, **bear**, and **sideways** — each with its own drift, volatility, and typical duration. The output feeds the Phase 1 HMM regime-detection work: since we know the ground-truth regime labels here, we can check whether the HMM actually recovers them before trusting it on real data.

Output: `data/synthetic_nq.csv` (gitignored — regenerate by re-running this notebook).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

RNG_SEED = 18
rng = np.random.default_rng(RNG_SEED)

## Regime parameters

| Regime   | Annual drift | Annual vol | Mean duration (days) |
|----------|-------------:|-----------:|----------------------:|
| Bull     | +20%         | 15%        | 90 (~4.5 months)      |
| Bear     | -30%         | 30%        | 45 (~2 months)        |
| Sideways | 0%           | 13%        | 60 (~3 months)        |

**Why these numbers:**

- **Bull drift/vol (+20%, 15%)** — approximates NQ's low-volatility grind-up phases (e.g. 2017, 2023–2024), where mega-cap tech growth drove a strong, relatively steady uptrend.
- **Bear drift/vol (-30%, 30%)** — approximates the 2022 Nasdaq-100 bear market (roughly -33% peak-to-trough) and the elevated realized volatility that accompanies equity drawdowns; NQ's vol regularly doubles or more during selloffs relative to calm bull markets.
- **Sideways drift/vol (0%, 13%)** — chop/consolidation with no sustained directional edge but still meaningful daily noise, sitting between bull and bear vol.
- **Bear regimes are shorter than bull regimes on average** — real bear markets and drawdowns tend to be sharper and faster than the bull runs that precede them (e.g. the 2020 COVID crash unwound in about a month; 2022's decline, while longer, moved in fast, punctuated legs).
- **Starting price of 15,000** reflects roughly where NQ futures have traded in the 2023–2025 window.

In [ ]:
TRADING_DAYS_PER_YEAR = 252

# annual_drift / annual_vol: approximate historical NQ behavior in each regime.
# mean_duration / duration_shape: parameters of the Gamma distribution used to
# sample how long (in trading days) a single visit to the regime lasts.
REGIMES = {
    "bull": {
        "annual_drift": 0.20,
        "annual_vol": 0.15,
        "mean_duration": 90,
        "duration_shape": 3.0,
    },
    "bear": {
        "annual_drift": -0.30,
        "annual_vol": 0.30,
        "mean_duration": 45,
        "duration_shape": 2.5,
    },
    "sideways": {
        "annual_drift": 0.0,
        "annual_vol": 0.13,
        "mean_duration": 60,
        "duration_shape": 3.0,
    },
}

# A regime never "self-transitions" here because its dwell time is already
# modeled by the Gamma-distributed duration draw. Momentum regimes are more
# likely to cool into sideways consolidation before flipping to the opposite
# regime, mirroring how real market cycles usually don't reverse instantly.
TRANSITION_PROBS = {
    "bull":     {"sideways": 0.65, "bear": 0.35},
    "bear":     {"sideways": 0.65, "bull": 0.35},
    "sideways": {"bull": 0.5, "bear": 0.5},
}

N_DAYS = TRADING_DAYS_PER_YEAR * 5  # 5 years of daily bars
START_PRICE = 15_000.0              # roughly where NQ has traded in 2023-2025
START_DATE = "2020-01-02"

## Why Gamma-distributed durations, not uniform

A uniform duration would imply every bull run or selloff lasts roughly the same number of days, which isn't how markets behave — real regimes are irregular, with many medium-length periods and occasional long outliers.

We instead draw each regime's duration from a **Gamma distribution** (shape > 1, scaled to the target mean), which is:

- **Right-skewed and strictly positive** — matching the empirical shape of regime/trend lengths.
- **A close cousin of the geometric/exponential sojourn times a hidden Markov chain naturally produces** — since Phase 1 will fit an HMM to this data, using Gamma-like dwell times here is a reasonably honest test bed, not one that's artificially easy (uniform) or artificially memoryless in a way that produces too many implausibly short regimes (pure exponential with shape=1).
- **Floored at 5 trading days** so we never generate a "regime" lasting 1–2 days, which wouldn't be tradable or meaningful.

Regime transitions also avoid snapping directly between bull and bear: momentum regimes pass through a sideways/consolidation phase most of the time before reversing, similar to how real market cycles usually cool off before turning.

In [ ]:
def sample_duration(shape, mean, rng, min_days=5):
    """Gamma-distributed regime duration (right-skewed, always positive)."""
    scale = mean / shape
    duration = rng.gamma(shape, scale)
    return max(int(round(duration)), min_days)


def build_regime_path(n_days, rng, start_regime="sideways"):
    path = []
    current = start_regime
    while len(path) < n_days:
        duration = sample_duration(
            REGIMES[current]["duration_shape"],
            REGIMES[current]["mean_duration"],
            rng,
        )
        path.extend([current] * duration)
        next_options = list(TRANSITION_PROBS[current].keys())
        next_probs = list(TRANSITION_PROBS[current].values())
        current = rng.choice(next_options, p=next_probs)
    return path[:n_days]


regime_path = build_regime_path(N_DAYS, rng)
pd.Series(regime_path).value_counts()

## Why Student-t innovations, not Gaussian

Equity index futures returns — NQ included — are well known to have fatter tails than a normal distribution implies (occasional large one-day moves well beyond what Gaussian vol would predict). We draw shocks from a Student-t distribution with 5 degrees of freedom, scaled so its variance still matches the target daily vol for that regime. This keeps the per-regime volatility calibration correct while adding realistic occasional large moves — and previews Phase 1's move from a Gaussian HMM baseline to a more robust skew-t emission model.

In [ ]:
def simulate_log_returns(regime_path, rng, t_dof=5):
    """
    Draw daily log returns per regime using Student-t innovations instead of
    Gaussian ones, since NQ (and equity index futures generally) exhibit
    fatter tails and occasional large jumps than a normal distribution
    implies. t_dof=5 gives noticeably fat tails while keeping variance finite.
    """
    n = len(regime_path)
    regime_arr = np.array(regime_path)
    log_returns = np.empty(n)

    for regime_name, params in REGIMES.items():
        mask = regime_arr == regime_name
        n_r = mask.sum()
        if n_r == 0:
            continue
        daily_drift = params["annual_drift"] / TRADING_DAYS_PER_YEAR
        daily_vol = params["annual_vol"] / np.sqrt(TRADING_DAYS_PER_YEAR)
        # scale t-draws so their variance matches daily_vol**2
        t_scale = daily_vol * np.sqrt((t_dof - 2) / t_dof)
        shocks = rng.standard_t(t_dof, size=n_r) * t_scale
        log_returns[mask] = daily_drift + shocks

    return log_returns


log_returns = simulate_log_returns(regime_path, rng)
close = START_PRICE * np.exp(np.cumsum(log_returns))

## From daily returns to OHLC bars

The close series comes directly from the cumulative log returns above. Open, high, and low are synthesized around each close: the overnight gap and the intrabar range both scale with that day's regime volatility, so bear-regime bars are wider and gappier than sideways-regime bars, matching how realized ranges expand during selloffs. All four price fields are then snapped to a 0.25-point grid, matching the real minimum tick size of the CME NQ contract.

In [ ]:
def round_to_tick(price, tick=0.25):
    """NQ futures trade in 0.25-point ticks on CME; snap synthetic prices to that grid."""
    return np.round(price / tick) * tick


def synthesize_ohlc(close, regime_path, rng):
    """
    Build plausible open/high/low bars around each close. The overnight gap
    and intrabar range both scale with that day's regime volatility, so bear
    regimes (higher vol) produce wider, gappier bars than sideways regimes.
    """
    n = len(close)
    prev_close = np.concatenate(([close[0]], close[:-1]))
    open_ = np.empty(n)
    high = np.empty(n)
    low = np.empty(n)

    for i in range(n):
        daily_vol = REGIMES[regime_path[i]]["annual_vol"] / np.sqrt(TRADING_DAYS_PER_YEAR)
        gap = rng.normal(0, daily_vol * 0.3)
        open_[i] = prev_close[i] * (1 + gap)
        intrabar_range = abs(rng.normal(0, daily_vol * 0.8))
        high[i] = max(open_[i], close[i]) * (1 + intrabar_range)
        low[i] = min(open_[i], close[i]) * (1 - intrabar_range)

    return open_, high, low


open_, high, low = synthesize_ohlc(close, regime_path, rng)

dates = pd.bdate_range(start=START_DATE, periods=N_DAYS)

df = pd.DataFrame({
    "date": dates,
    "open": round_to_tick(open_),
    "high": round_to_tick(high),
    "low": round_to_tick(low),
    "close": round_to_tick(close),
    "log_return": log_returns,
    "regime": regime_path,
}).set_index("date")

df.head()

In [ ]:
output_path = Path("../data/synthetic_nq.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path)
print(f"Saved {len(df)} rows to {output_path.resolve()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = {"bull": "#2ca02c", "bear": "#d62728", "sideways": "#7f7f7f"}
regime_arr = np.array(regime_path)

change_points = np.where(regime_arr[:-1] != regime_arr[1:])[0] + 1
segment_starts = np.concatenate(([0], change_points))
segment_ends = np.concatenate((change_points, [len(regime_arr)]))

for s, e in zip(segment_starts, segment_ends):
    ax.axvspan(df.index[s], df.index[e - 1], color=colors[regime_arr[s]], alpha=0.15)

ax.plot(df.index, df["close"], color="black", linewidth=1)
ax.set_title("Synthetic NQ Futures Price with Regime Shading")
ax.set_ylabel("Price")

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c, alpha=0.3, label=r) for r, c in colors.items()])
plt.show()

## Checking the simulation against its own targets

The cells below compare what actually came out of one random draw against the
target parameters set above. Expect some drift — this is a single 5-year
realization, and with fat-tailed (Student-t) shocks the sample *mean* return
in a regime is noisy even over a few hundred days, especially for bear
regimes, which are both shorter and higher-vol. Realized *volatility*
converges much faster and should track the targets closely; realized *drift*
can plausibly be off by several points without indicating a bug. This mirrors
real markets, where a single historical bear market's realized return is not
a precise estimate of "the" bear-market drift either.

In [ ]:
# Sanity check: realized annualized drift/vol per regime vs. the targets above
summary = df.groupby("regime")["log_return"].agg(["mean", "std", "count"])
summary["annualized_drift"] = summary["mean"] * TRADING_DAYS_PER_YEAR
summary["annualized_vol"] = summary["std"] * np.sqrt(TRADING_DAYS_PER_YEAR)
summary[["count", "annualized_drift", "annualized_vol"]]

In [ ]:
# Sanity check: actual sampled regime durations vs. the target means
segment_lengths = pd.DataFrame({
    "regime": [regime_arr[s] for s in segment_starts],
    "duration": segment_ends - segment_starts,
})
segment_lengths.groupby("regime")["duration"].agg(["mean", "min", "max", "count"])

## Next steps

This dataset has known ground-truth regime labels, which real market data never does. Use it to validate the Phase 1 HMM before trusting it on historical or live NQ data:

1. Fit a Gaussian-emission HMM on `log_return` and check whether the recovered hidden states line up with the `regime` column.
2. Compare against a skew-t emission HMM to see whether it better captures the fat-tailed, regime-dependent volatility built into this data.